In [1]:
# === Core Python ===
import collections
import glob
import json
import logging
import os
import time
from datetime import datetime

# === Numerical and Data Handling ===
import numpy as np
import pandas as pd
import xarray as xr
#import xcdat as xcd
import xskillscore as xs
from scipy.signal import butter, filtfilt, lfilter, sosfilt
from scipy.stats import pearsonr
from skimage.feature import peak_local_max
import metpy.calc as mpcalc

# === Plotting ===
import matplotlib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.pylab import rcParams
from matplotlib.ticker import MultipleLocator, ScalarFormatter

# === Mapping and Geospatial ===
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as sticker

# === Climate Visualization Tools ===
import cmaps as gvcmaps
import geocat.viz as gv
import geocat.viz.util as gvutil

# === Tropical Cyclone Analysis ===
from tropycal import tracks, utils

# === Climate Dataset Access ===
import climetlab as cml

In [2]:
class ErrorMetricCalculator:
    def __init__(self, regnam, tstart, tend, frequency,
                 model_list, ref_dict, exp_dict, path_in, out_path,
                 var_list=None, force=False):
        self.regnam = regnam
        self.tstart = tstart
        self.tend = tend
        self.frequency = frequency
        self.model_list = model_list
        self.ref_dict = ref_dict
        self.exp_dict = exp_dict
        self.path_in = path_in
        self.out_path = os.path.join(out_path, frequency)
        self.force = force

        self.var_dict = self.extract_var_list()
        self.var_list = var_list if var_list is not None else list(self.var_dict.keys())
        self.ref_cache = {}
        self.years = list(range(int(self.tstart[:4]), int(self.tend[:4]) + 1))

        self.seasons = {
            "DJF": ([12, 1, 2], "01-03"),
            "MAM": ([3, 4, 5], "04-06"),
            "JJA": ([6, 7, 8], "07-09"),
            "SON": ([9, 10, 11], "10-12"),
        }

        if not os.path.exists(self.out_path):
            os.makedirs(self.out_path)

    def compute(self, metric):
        for year in self.years:
            for var in self.var_list:
                if var not in self.var_dict:
                    raise ValueError(f"Variable '{var}' is not defined in var_dict.")
                vinfo = self.var_dict[var]
                self._compute_seasonal_and_annual_metric(metric, var, vinfo, year)

    def _compute_seasonal_and_annual_metric(self, metric, var, vinfo, year):
        varin = vinfo['alias']
        vfac = vinfo['fscl']
        season_list = ["DJF", "MAM", "JJA", "SON", "ANN"]

        for exp in self.model_list:
            ref = self.exp_dict[exp]['ref']
            out_file = os.path.join(self.out_path, f"{var}_{self.regnam}_{metric}_{exp}_{year}.nc")

            if os.path.exists(out_file) and not self.force:
                print(f"Skipping {metric} for {var} in {exp} ({year}) — file exists.")
                continue

            results = {}
            for season in season_list:
                time_sub = self._get_time_range_for_year(year) if season == "ANN" else self._get_time_range_for_season(year, season)
                try:
                    obs = self._read_reference_data(
                        ref=ref,
                        period=f"{year}01-{year}12",
                        time_sub=time_sub,
                        regnam=self.regnam,
                        var=var,
                        vfac=vfac,
                        data_dir=self.ref_dict[ref]['run'],
                        template="6hourly/ERA5.6hourly.en00.{}.{}.nc"
                    )[varin].astype("float64")

                    ds = self._read_model_data(
                        exp=exp,
                        period=f"{year}01-{year}12",
                        time_sub=time_sub,
                        regnam=self.regnam,
                        var=var,
                        vfac=vfac,
                        data_dir=self.path_in.replace("%(CASENAME)", self.exp_dict[exp]['run']),
                        template="{}_{}_*.nc"
                    )
                    ds = ds.assign_coords(time=obs['time'])
                    fcst = ds[varin].astype("float64")
                except FileNotFoundError:
                    print(f"Data missing for {var}, {exp}, {year}, {season} — skipping.")
                    results[season] = np.nan
                    continue

                if metric == "MEAN":
                    val = fcst.weighted(self.generate_coslat_weight(fcst)).mean(("lat", "lon", "time")).values
                elif metric == "BIAS":
                    val = (fcst - obs).weighted(self.generate_coslat_weight(fcst)).mean(("lat", "lon", "time")).values
                elif metric == "RMSE":
                    weights = self.generate_coslat_weight(obs)[0, :, :]
                    val = xs.rmse(obs, fcst, dim=["lat", "lon"], weights=weights, skipna=True).mean("time").values
                elif metric == "ACC":
                    weights = self.generate_coslat_weight(obs)[0, :, :]
                    acc_map = xs.pearson_r(obs - obs.mean("time"), fcst - fcst.mean("time"),
                                           dim="time", skipna=True)
                    val = (acc_map * weights).mean(["lat", "lon"]).values
                elif metric == "STD":
                    val = fcst.std("time").weighted(self.generate_coslat_weight(fcst)).mean(("lat", "lon")).values
                elif metric == "TSCORR":
                    ts_corrs = []
                    for t in range(fcst.time.size):
                        f_slice = fcst.isel(time=t)
                        o_slice = obs.isel(time=t)
                        corr = xs.pearson_r(o_slice, f_slice, dim=["lat", "lon"], skipna=True).values
                        ts_corrs.append(corr)
                    val = np.nanmean(ts_corrs)
                elif metric == "STCORR":
                    corr = xs.pearson_r(obs, fcst, dim="time", skipna=True)
                    weights = self.generate_coslat_weight(fcst)[0, :, :]
                    val = (corr * weights).mean(["lat", "lon"]).values
                else:
                    raise ValueError(f"Unsupported metric: {metric}")

                results[season] = val

            ds_out = xr.Dataset({
                var: xr.DataArray(np.asarray([results[s] for s in season_list]),
                                  dims="season", coords={"season": season_list})
            })
            ds_out.to_netcdf(out_file)

    def _get_time_range_for_year(self, year):
        freq_map = {"3hourly": "3h", "6hourly": "6h", "monthly": "1MS"}
        return xr.cftime_range(f"{year}-01-01", f"{year}-12-31",
                               freq=freq_map[self.frequency], calendar="noleap")

    def _get_time_range_for_season(self, year, season):
        from cftime import DatetimeNoLeap
        months, _ = self.seasons[season]
        start_year = year if 1 in months else year - 1
        end_year = year if 12 not in months else year + 1
        return xr.cftime_range(
            start=DatetimeNoLeap(start_year, months[0], 1),
            end=DatetimeNoLeap(end_year, months[-1], 28),
            freq={"3hourly": "3h", "6hourly": "6h", "monthly": "1MS"}[self.frequency],
            calendar="noleap"
        )

    def _read_model_data(self, exp, period, time_sub, regnam, var, vfac, data_dir, template):
        var_file = os.path.join(data_dir, template.format(var, period[0:4]))
        file_list = glob.glob(var_file)
        if not file_list:
            raise FileNotFoundError(f"No data found for {exp} at {var_file}")

        region = self.define_region(regnam)
        lat_slice = slice(region[0][0], region[0][1])
        lon_slice = slice(region[1][0], region[1][1])

        dm = xr.open_dataset(file_list[0])
        for lev_dim in ["lev", "plev", "level"]:
            if lev_dim in dm.dims:
                dm = dm.isel({lev_dim: 0}, drop=True)
        if dm.lon.min() > -1.0:
            dm = dm.assign_coords(lon=((dm.lon + 180) % 360 - 180)).sortby('lon')
        dm = dm.convert_calendar("noleap", use_cftime=True)
        ds = dm.sel(time=time_sub, method='nearest')
        ds = ds.sel(lat=lat_slice, lon=lon_slice)
        ds[var] = self.apply_unit_scaling_mod(var, ds[var], vfac)
        return ds

    def _read_reference_data(self, ref, period, time_sub, regnam, var, vfac, data_dir, template):
        rpath = os.path.join(data_dir, template.format(var, period))
        dr = xr.open_dataset(rpath).rename({'longitude': 'lon', 'latitude': 'lat'})
        for lev_dim in ["lev", "plev", "level"]:
            if lev_dim in dr.dims:
                dr = dr.isel({lev_dim: 0}, drop=True)
        if dr.lon.min() > -1.0:
            dr = dr.assign_coords(lon=((dr.lon + 180) % 360 - 180)).sortby('lon')
        dr = dr.convert_calendar("noleap", use_cftime=True)
        region = self.define_region(regnam)
        dr = dr.sel(time=time_sub, method='nearest')
        dr = dr.sel(lat=slice(*region[0]), lon=slice(*region[1]))
        dr[var] = self.apply_unit_scaling_obs(var, dr[var], vfac)
        return dr

    @staticmethod
    def generate_coslat_weight(ds):
        weights = np.cos(np.deg2rad(ds.lat))
        _, weights = xr.broadcast(ds, weights)
        return weights

    @staticmethod
    def apply_unit_scaling_obs(var, da, vfac):
        if var in ['Z200', 'Z500', 'Z850']:
            return da * vfac / 9.80616
        elif var in ['SHFLX', 'TAUX', 'TAUY']:
            return da * vfac * -1.0
        elif var == 'LHFLX':
            return da * vfac * -2.501e6
        elif var in ['T200', 'T500', 'T850', 'TREFHT', 'TS']:
            return (da - 273.15) * vfac
        elif var == 'PRECT':
            return da * vfac / 3600.0 * 1000.0 * 86400.0
        else:
            return da * vfac

    @staticmethod
    def apply_unit_scaling_mod(var, da, vfac):
        if var in ['Z200', 'Z500', 'Z850']:
            return da * vfac / 9.80616
        elif var == 'PRECT':
            return da * vfac * 1000.0 * 86400.0
        elif var == 'LHFLX':
            return da * vfac * -2.501e6
        elif var in ['T200', 'T500', 'T850', 'TREFHT', 'TS']:
            return (da - 273.15) * vfac
        else:
            return da * vfac

    @staticmethod
    def define_region(regnam='global'):
        reg_dict = {
            'global': [(-90, 90), (-180, 180)],
            'Atlantic': [(5, 55), (-95, -40)],
            'CONUS': [(25, 50), (-125, -95)],
            'Antarctic': [(-90, -50), (-180, 180)],
            'PolarN': [(50, 90), (-180, 180)],
            'Greenland': [(60, 85), (-75, -10)]
        }
        return reg_dict[regnam]

    @staticmethod
    def extract_var_list():
        return {
            'U200':       {'alias': 'U200',     'unit': 'm s$^{-1}$',           'fscl': 1.0,    'min': -0.2, 'max': 0.2, 'nlev': 11},
            'U500':       {'alias': 'U500',     'unit': 'm s$^{-1}$',           'fscl': 1.0,    'min': -0.2, 'max': 0.2, 'nlev': 11},
            'U850':       {'alias': 'U850',     'unit': 'm s$^{-1}$',           'fscl': 1.0,    'min': -0.2, 'max': 0.2, 'nlev': 11},
            'V200':       {'alias': 'V200',     'unit': 'm s$^{-1}$',           'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'V500':       {'alias': 'V500',     'unit': 'm s$^{-1}$',           'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'V850':       {'alias': 'V850',     'unit': 'm s$^{-1}$',           'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'T850':       {'alias': 'T850',     'unit': '$^{o}$C',              'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'T200':       {'alias': 'T200',     'unit': '$^{o}$C',              'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'T500':       {'alias': 'T500',     'unit': '$^{o}$C',              'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
            'Q850':       {'alias': 'Q850',     'unit': 'g kg$^{-1}$',          'fscl': 1e3,    'min': -2,   'max': 2,   'nlev': 11},
            'Q200':       {'alias': 'Q200',     'unit': 'g kg$^{-1}$',          'fscl': 1e3,    'min': -2,   'max': 2,   'nlev': 11},
            'Q500':       {'alias': 'Q500',     'unit': 'g kg$^{-1}$',          'fscl': 1e3,    'min': -2,   'max': 2,   'nlev': 11},
            'OMEGA850':   {'alias': 'OMEGA850', 'unit': 'g kg$^{-1}$',          'fscl': 1e3,    'min': -2,   'max': 2,   'nlev': 11},
            'OMEGA200':   {'alias': 'OMEGA200', 'unit': 'g kg$^{-1}$',          'fscl': 1e3,    'min': -2,   'max': 2,   'nlev': 11},
            'OMEGA500':   {'alias': 'OMEGA500', 'unit': 'g kg$^{-1}$',          'fscl': 1e3,    'min': -2,   'max': 2,   'nlev': 11},            
            'Z850':       {'alias': 'Z850',     'unit': 'hectometer',           'fscl': 1e-2,   'min': -2,   'max': 2,   'nlev': 11},
            'Z200':       {'alias': 'Z200',     'unit': 'hectometer',           'fscl': 1e-2,   'min': -2,   'max': 2,   'nlev': 11},
            'Z500':       {'alias': 'Z500',     'unit': 'hectometer',           'fscl': 1e-2,   'min': -2,   'max': 2,   'nlev': 11},
            'PSL':        {'alias': 'PSL',      'unit': 'hPa',                  'fscl': 1e-2,   'min': -2,   'max': 2,   'nlev': 11},
            'PRECT':      {'alias': 'PRECT',    'unit': 'mm day$^{-1}$',        'fscl': 1.0,    'min': -2,   'max': 2,   'nlev': 11},
        }

In [3]:
if __name__ == "__main__":
    # Set basic paths
    top_path  = "/anvil/scratch/x-szhang3"
    data_path = f"{top_path}/post_data"
    out_path  = "/home/x-szhang3/nudging_analysis/nudging_evaluation/data/error_metrics"
    os.makedirs(out_path, exist_ok=True)

    # Load model experiment metadata
    exp_json = f"/home/x-szhang3/nudging_analysis/nudge_exp_info.json"
    with open(exp_json, "r") as f:
        exp_dict = json.load(f)
        
    exp_to_remove = "NDGUVTQ_SRF2"
    if exp_to_remove in exp_dict:
        del exp_dict[exp_to_remove]
        
    # Define reference (ERA5) metadata
    ref_dict = {
        "ERA5": {
            "run": "/anvil/scratch/x-szhang3/post_data/ERA5",
            "period": "200801_201712"
        }
    }

    # Time and frequency
    tstart = "2008-01"
    tend   = "2017-12"
    freq   = "6hourly"
    regnam = "global"
    
    # Template for locating model files
    path_template = (
        f"/anvil/scratch/x-szhang3/post_data/%(CASENAME)/{freq}"
    )

    # Metrics and variables to compute
    #metrics = ["MEAN", "BIAS", "RMSE", "ACC", "TSCORR", "STCORR"]
    metrics = ["MEAN", "BIAS", "TSCORR", "STCORR"]
    variables = None #["T850", "PRECT"]

    # Initialize and run
    calculator = ErrorMetricCalculator(
        regnam=regnam,
        tstart=tstart,
        tend=tend,
        frequency=freq,
        model_list=list(exp_dict.keys()),
        ref_dict=ref_dict,
        exp_dict=exp_dict,
        path_in=path_template,
        out_path=out_path,
        var_list=variables,
        force=False
    )

    for metric in metrics:
        calculator.compute(metric)

PermissionError: [Errno 13] Permission denied: '/home/x-szhang3'